In [ ]:
#Similar product Recommendation System

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import re 

In [8]:
df=pd.read_excel("D:\\inventory recommendation\\data set\\Merged file.xlsx")
df_trans=pd.read_excel("D:\\inventory recommendation\\data set\\transaction_dataset_1200.xlsx")
print(df.head())

  product_id                   product_name                category  \
0     BPC001  L'Oreal Revitalift Face Cream  Beauty & Personal Care   
1     BPC002                   Dove Shampoo  Beauty & Personal Care   
2     BPC003    Neutrogena Sunscreen SPF 30  Beauty & Personal Care   
3     BPC004                 Nivea Lip Balm  Beauty & Personal Care   
4     BPC005    The Body Shop Aloe Vera Gel  Beauty & Personal Care   

           brand  price  rating  num_reviews  \
0        L'Oreal    899     4.6          650   
1           Dove    299     4.3          800   
2     Neutrogena    799     4.7          900   
3          Nivea    199     4.5          700   
4  The Body Shop    699     4.4          540   

                                  description        type  \
0     Anti-aging face cream with Pro-Retinol.  Face Cream   
1          Moisturizing shampoo for dry hair.     Shampoo   
2            Broad spectrum sunscreen SPF 30.   Sunscreen   
3     Moisturizing lip balm with shea bu

In [9]:
def clean_text(text):
    if isinstance(text,str):
        text=text.lower()
        text=re.sub(r'[^a-z0-9\s]','',text)
        return text
    return ""

In [10]:
df['text_features']=(df['product_name'].fillna('')+' '+df["category"].fillna("")+' '+df['brand'].fillna('')+" "+df['type'].fillna("")+" "+df['description'].fillna('')).apply(clean_text)

print(df["text_features"])


0      loreal revitalift face cream beauty  personal ...
1      dove shampoo beauty  personal care dove shampo...
2      neutrogena sunscreen spf 30 beauty  personal c...
3      nivea lip balm beauty  personal care nivea lip...
4      the body shop aloe vera gel beauty  personal c...
                             ...                        
355    saffola butter 56 groceries saffola butter saf...
356    24 mantra spice 57 groceries 24 mantra spice 2...
357    dhampure snack 58 groceries dhampure snack dha...
358    tata salt tea 59 groceries tata salt tea tata ...
359    indiagate rice 60 groceries indiagate rice ind...
Name: text_features, Length: 360, dtype: object


In [11]:
vector=TfidfVectorizer(stop_words='english')
tfidf_matrix=vector.fit_transform(df['text_features'])
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3250 stored elements and shape (360, 501)>
  Coords	Values
  (0, 284)	0.41818241840245596
  (0, 377)	0.24538072768080169
  (0, 181)	0.5137450857264438
  (0, 139)	0.587295872648919
  (0, 92)	0.11002231868605308
  (0, 344)	0.10690221714705783
  (0, 117)	0.11002231868605308
  (0, 75)	0.24538072768080169
  (0, 362)	0.24538072768080169
  (1, 92)	0.1362579245005789
  (1, 344)	0.1323938125184666
  (1, 117)	0.1362579245005789
  (1, 159)	0.5397908677841597
  (1, 398)	0.7076920716514283
  (1, 305)	0.28400591061553615
  (1, 163)	0.20189928054220574
  (1, 226)	0.17808567729867358
  (2, 92)	0.10602021212285535
  (2, 344)	0.103013605545575
  (2, 117)	0.10602021212285535
  (2, 309)	0.42000303845970927
  (2, 443)	0.5659327465619833
  (2, 428)	0.47290980794203197
  (2, 25)	0.36709626897738645
  (2, 108)	0.23645490397101598
  :	:
  (357, 234)	0.10289656931855194
  (357, 223)	0.10044430737693895
  (357, 153)	0.10353653301275662
  (357, 149)	0.

In [12]:
cosine=cosine_similarity(tfidf_matrix,tfidf_matrix)
print(cosine)

[[1.         0.04413602 0.03434156 ... 0.         0.         0.        ]
 [0.04413602 1.         0.04253055 ... 0.         0.         0.        ]
 [0.03434156 0.04253055 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.04485959 0.04939246]
 [0.         0.         0.         ... 0.04485959 1.         0.04584584]
 [0.         0.         0.         ... 0.04939246 0.04584584 1.        ]]


In [13]:
def similar_products(product_name,top_n=10):
    idx=df.index[df['product_name'].str.lower()==product_name.lower()].tolist()
    if not idx:
        return "product'{product_name}'not fund in dataset"
    idx=idx[0]
    sim_scores=list(enumerate(cosine[idx]))
    sim_scores=sorted(sim_scores,key=lambda x: x[1],reverse=True)
    top_products=sim_scores[1:top_n+1]
    product=[i[0] for i in top_products]
    result=df.loc[product,['product_id',"product_name","category",'brand','price','rating','features']].copy()
    result['similarity_score']=[i[1] for i in top_products]
    return result.reset_index(drop=True)

In [ ]:
example_product="Kama Ayurveda Lip Balm 13"
similar_items=similar_products(example_product,top_n=10)
print(similar_items)

  product_id               product_name                category  \
0     BPC053  Kama Ayurveda Lip Balm 53  Beauty & Personal Care   
1     BPC033  Kama Ayurveda Lip Balm 33  Beauty & Personal Care   
2     BPC043          Nivea Lip Balm 43  Beauty & Personal Care   
3     BPC023          Nivea Lip Balm 23  Beauty & Personal Care   
4     BPC004             Nivea Lip Balm  Beauty & Personal Care   

           brand  price  rating  \
0  Kama Ayurveda   2150     4.3   
1  Kama Ayurveda   2150     3.8   
2          Nivea    650     4.8   
3          Nivea    650     4.3   
4          Nivea    199     4.5   

                                            features  similarity_score  
0  {"quantity": "2pcs", "ingredients": "Charcoal,...          0.913778  
1  {"quantity": "30ml", "ingredients": "Charcoal,...          0.913464  
2  {"quantity": "1.2g", "ingredients": "Charcoal,...          0.617642  
3  {"quantity": "10g", "ingredients": "Charcoal, ...          0.617507  
4  {"quantity": "10g"

In [15]:
import joblib

class SimilarProductModel:
    def __init__(self, df, cosine_matrix):
        self.df = df.reset_index(drop=True)
        self.cosine = cosine_matrix
        
    def predict(self, product_name, top_n=10):
        idx = self.df.index[self.df['product_name'].str.lower() == product_name.lower()].tolist()
        if not idx:
            return []
        
        idx = idx[0]
        sim_scores = list(enumerate(self.cosine[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        top_products = sim_scores[1:top_n+1]
        product_idx = [i[0] for i in top_products]
        
        result = self.df.loc[product_idx, [
            'product_id', 'product_name', 'category', 'brand',
            'price', 'rating', 'features'
        ]].copy()
        
        result['similarity_score'] = [i[1] for i in top_products]
        return result.reset_index(drop=True)



In [16]:
similar_model = SimilarProductModel(df, cosine)


In [17]:
joblib.dump(similar_model, "similar_model.pkl")


['similar_model.pkl']